In [ ]:
!pip install -q datasets transformers scikit-learn xgboost

In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

dataset = load_dataset("ailsntua/QEvasion")
train_df_full = dataset['train'].to_pandas()

train_df, val_df = train_test_split(
    train_df_full,
    test_size=750,
    stratify=train_df_full['evasion_label'],
    random_state=42
)

X_train = train_df['question'] + ' ' + train_df['interview_answer']
y_train = train_df['evasion_label']

X_test = val_df['question'] + ' ' + val_df['interview_answer']
y_test = val_df['evasion_label']

globals().update({
    'X_train': X_train,
    'y_train': y_train,
    'X_test': X_test,
    'y_test': y_test,
    'train_df': train_df,
    'val_df': val_df
})

In [ ]:
print("Logistic Regression Subtask 2 - Evasion Classification:\n")


baseline_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',      
        ngram_range=(1, 2),         
        max_features=20000,
        min_df=2,                   
        max_df=0.85,                
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
        C=10.0
    ))
])
#best parameters according to grid search

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"Macro F1: {macro_f1:.4f}\n")
print(classification_report(y_test, y_pred, zero_division=0))

Logistic Regression Subtask 2 - Evasion Classification:

Macro F1: 0.2905

                     precision    recall  f1-score   support

   Claims ignorance       0.22      0.27      0.24        26
      Clarification       0.43      0.65      0.52        20
Declining to answer       0.28      0.42      0.34        31
         Deflection       0.22      0.35      0.27        83
            Dodging       0.30      0.30      0.30       154
           Explicit       0.48      0.28      0.35       229
            General       0.24      0.21      0.23        84
           Implicit       0.20      0.25      0.22       106
Partial/half-answer       0.12      0.18      0.15        17

           accuracy                           0.29       750
          macro avg       0.28      0.32      0.29       750
       weighted avg       0.32      0.29      0.29       750

Macro F1: 0.2905

                     precision    recall  f1-score   support

   Claims ignorance       0.22      0.27      0.2

In [ ]:
print("Logistic Regression Subtask 1 - Evasion-Based Clarity Evaluation\n")

CLARITY_MAPPING = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "Dodging": "Ambivalent",
    "General": "Ambivalent",
    "Deflection": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply"
}

y_pred_clarity = [CLARITY_MAPPING[label] for label in y_pred]
y_test_clarity = [CLARITY_MAPPING[label] for label in y_test]

clarity_f1 = f1_score(y_test_clarity, y_pred_clarity, average='macro')

print(f"Clarity Macro F1: {clarity_f1:.4f}\n")
print(classification_report(y_test_clarity, y_pred_clarity, zero_division=0))

🔍 Logistic Regression Subtask 1 - Evasion-Based Clarity Evaluation

Clarity Macro F1: 0.4877

                 precision    recall  f1-score   support

     Ambivalent       0.66      0.75      0.70       444
Clear Non-Reply       0.35      0.47      0.40        77
    Clear Reply       0.47      0.29      0.36       229

       accuracy                           0.58       750
      macro avg       0.49      0.50      0.49       750
   weighted avg       0.57      0.58      0.57       750



In [ ]:
from sklearn.model_selection import GridSearchCV

optimized_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidf__max_features': [10000, 15000, 17500, 20000],
    'tfidf__min_df': [1, 2, 3, 5],
    'tfidf__sublinear_tf': [True, False],
    'clf__C': [0.1, 1.0, 10.0],
    'clf__solver': ['lbfgs', 'saga']
}

grid_search = GridSearchCV(
    optimized_pipeline,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

y_pred_optimized = grid_search.predict(X_test)
macro_f1_optimized = f1_score(y_test, y_pred_optimized, average='macro')

for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")
print(f"\nCV score: {grid_search.best_score_:.4f}")
print(f"Test score: {macro_f1_optimized:.4f}")
print(classification_report(y_test, y_pred_optimized, zero_division=0))

In [21]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

def create_features(df):
    text = df['question'] + ' ' + df['interview_answer']
    
    if 'gpt3.5_summary' in df.columns:
        text = text + ' ' + df['gpt3.5_summary'].fillna('')
    
    numerical_features = {
        'answer_length': df['interview_answer'].str.len(),
        'question_length': df['question'].str.len(),
        'answer_words': df['interview_answer'].str.split().str.len(),
        'question_words': df['question'].str.split().str.len(),
        'ratio': df['interview_answer'].str.len() / (df['question'].str.len() + 1)
    }
    
    if 'inaudible' in df.columns:
        numerical_features['inaudible'] = df['inaudible'].astype(int)
    if 'multiple_questions' in df.columns:
        numerical_features['multiple_questions'] = df['multiple_questions'].astype(int)
    if 'affirmative_questions' in df.columns:
        numerical_features['affirmative_questions'] = df['affirmative_questions'].astype(int)
    
    numerical = pd.DataFrame(numerical_features)
    return text, numerical

X_train_text, X_train_num = create_features(train_df)
X_test_text, X_test_num = create_features(val_df)

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=20000, sublinear_tf=True, min_df=2, max_df=0.85)
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)

X_train_combined = hstack([X_train_tfidf, X_train_num_scaled])
X_test_combined = hstack([X_test_tfidf, X_test_num_scaled])

clf_enhanced = LogisticRegression(
    class_weight='balanced',
    C=1.0,
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)

clf_enhanced.fit(X_train_combined, y_train)

y_pred_enhanced = clf_enhanced.predict(X_test_combined)
macro_f1_enhanced = f1_score(y_test, y_pred_enhanced, average='macro')

print(f"\n Subtask 2 Evasion Classification: Macro F1 with Features: {macro_f1_enhanced:.4f}")
print(f"vs Baseline: +{(macro_f1_enhanced - macro_f1):.4f} ({((macro_f1_enhanced/macro_f1 - 1)*100):.1f}%)\n")
print(classification_report(y_test, y_pred_enhanced, zero_division=0))


 Macro F1 with Features: 0.3321
vs Baseline: +0.0416 (14.3%)

                     precision    recall  f1-score   support

   Claims ignorance       0.21      0.27      0.24        26
      Clarification       0.33      0.90      0.48        20
Declining to answer       0.53      0.32      0.40        31
         Deflection       0.24      0.34      0.28        83
            Dodging       0.47      0.31      0.37       154
           Explicit       0.56      0.50      0.53       229
            General       0.29      0.26      0.27        84
           Implicit       0.28      0.27      0.28       106
Partial/half-answer       0.10      0.24      0.14        17

           accuracy                           0.37       750
          macro avg       0.33      0.38      0.33       750
       weighted avg       0.41      0.37      0.38       750



In [22]:
print("Subtask 1 - Evasion-based Clarity Evaluation\n")

CLARITY_MAPPING = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "Dodging": "Ambivalent",
    "General": "Ambivalent",
    "Deflection": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply"
}

y_pred_clarity_enhanced = [CLARITY_MAPPING[label] for label in y_pred_enhanced]
y_test_clarity_enhanced = [CLARITY_MAPPING[label] for label in y_test]

clarity_f1_enhanced = f1_score(y_test_clarity_enhanced, y_pred_clarity_enhanced, average='macro')

print(f"Clarity Macro F1 with features: {clarity_f1_enhanced:.4f}")
print(f"vs Baseline Clarity: +{(clarity_f1_enhanced - clarity_f1):.4f} ({((clarity_f1_enhanced/clarity_f1 - 1)*100):.1f}%)\n")
print(classification_report(y_test_clarity_enhanced, y_pred_clarity_enhanced, zero_division=0))

Subtask 1 - Evasion-based Clarity Evaluation

Clarity Macro F1 (Enhanced Features): 0.5717
vs Baseline Clarity: +0.0840 (17.2%)

                 precision    recall  f1-score   support

     Ambivalent       0.72      0.72      0.72       444
Clear Non-Reply       0.40      0.56      0.47        77
    Clear Reply       0.56      0.50      0.53       229

       accuracy                           0.63       750
      macro avg       0.56      0.59      0.57       750
   weighted avg       0.64      0.63      0.63       750



In [ ]:
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

sample_weights = compute_sample_weight('balanced', y_train_encoded)

xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=9,
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train_combined, y_train_encoded, sample_weight=sample_weights, verbose=False)
y_pred_xgb_encoded = xgb_model.predict(X_test_combined)
y_pred_xgb = label_encoder.inverse_transform(y_pred_xgb_encoded)

macro_f1_xgb = f1_score(y_test, y_pred_xgb, average='macro')

print(f"\n Subtask 2 - Evasion Classification XGBoost Macro F1: {macro_f1_xgb:.4f}")
print(f" vs Baseline: +{(macro_f1_xgb - macro_f1):.4f} ({((macro_f1_xgb/macro_f1 - 1)*100):.1f}%)\n")
print(classification_report(y_test, y_pred_xgb, zero_division=0))

🚀 XGBoost Classifier

📊 Mapare clase:
   0: Claims ignorance
   1: Clarification
   2: Declining to answer
   3: Deflection
   4: Dodging
   5: Explicit
   6: General
   7: Implicit
   8: Partial/half-answer

⏳ Antrenare (2-3 minute)...

🏆 XGBoost Macro F1: 0.3852
📈 vs Baseline: +0.0947 (32.6%)

                     precision    recall  f1-score   support

   Claims ignorance       0.36      0.31      0.33        26
      Clarification       0.50      0.65      0.57        20
Declining to answer       0.55      0.19      0.29        31
         Deflection       0.28      0.35      0.31        83
            Dodging       0.47      0.47      0.47       154
           Explicit       0.62      0.67      0.64       229
            General       0.31      0.31      0.31        84
           Implicit       0.40      0.35      0.37       106
Partial/half-answer       0.33      0.12      0.17        17

           accuracy                           0.46       750
          macro avg       0.42

In [25]:
print("Subtask 1 - XGBoost Evasion-based Clarity Evaluation\n")

CLARITY_MAPPING = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "Dodging": "Ambivalent",
    "General": "Ambivalent",
    "Deflection": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply"
}

y_pred_clarity_xgb = [CLARITY_MAPPING[label] for label in y_pred_xgb]
y_test_clarity_xgb = [CLARITY_MAPPING[label] for label in y_test]

clarity_f1_xgb = f1_score(y_test_clarity_xgb, y_pred_clarity_xgb, average='macro')

print(f"Macro F1: {clarity_f1_xgb:.4f}")
print(f"vs Baseline Clarity: +{(clarity_f1_xgb - clarity_f1):.4f} ({((clarity_f1_xgb/clarity_f1 - 1)*100):.1f}%)")
print(f"vs Enhanced Features Clarity: +{(clarity_f1_xgb - clarity_f1_enhanced):.4f} ({((clarity_f1_xgb/clarity_f1_enhanced - 1)*100):.1f}%)\n")
print(classification_report(y_test_clarity_xgb, y_pred_clarity_xgb, zero_division=0))

Subtask 1 - XGBoost Evasion-based Clarity Evaluation

Macro F1: 0.6236
vs Baseline Clarity: +0.1359 (27.9%)
vs Enhanced Features Clarity: +0.0519 (9.1%)

                 precision    recall  f1-score   support

     Ambivalent       0.76      0.75      0.76       444
Clear Non-Reply       0.54      0.42      0.47        77
    Clear Reply       0.62      0.67      0.64       229

       accuracy                           0.69       750
      macro avg       0.64      0.61      0.62       750
   weighted avg       0.69      0.69      0.69       750

